In [69]:
# Install required packages (run once)
!pip install -q torch

In [70]:
!pip install -q torch-geometric


In [71]:
!pip install -q energyflow matplotlib seaborn scikit-learn tqdm

In [5]:
import torch
import os

# GPU Optimization Settings
print("=" * 80)
print("GPU CONFIGURATION")
print("=" * 80)

# Check CUDA availability
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
    
    # Set default GPU (use GPU 0 or specify CUDA_VISIBLE_DEVICES)
    device = torch.device('cuda:0')
    print(f"\nUsing device: {device}")
    
    # Enable optimizations for A100
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print("\nOptimizations enabled:")
    print("  - TF32 for matmul: True")
    print("  - TF32 for cuDNN: True")
    print("  - cuDNN benchmark: True")
else:
    device = torch.device('cpu')
    print("\nWARNING: CUDA not available, using CPU")

print("=" * 80)

GPU CONFIGURATION
PyTorch Version: 2.10.0+cu128
CUDA Available: True
CUDA Version: 12.8
Number of GPUs: 4

GPU 0: NVIDIA A100-PCIE-40GB
  Memory: 42.29 GB

GPU 1: NVIDIA A100-PCIE-40GB
  Memory: 42.29 GB

GPU 2: NVIDIA A100-PCIE-40GB
  Memory: 42.29 GB

GPU 3: NVIDIA A100-PCIE-40GB
  Memory: 42.29 GB

Using device: cuda:0

Optimizations enabled:
  - TF32 for matmul: True
  - TF32 for cuDNN: True
  - cuDNN benchmark: True


In [6]:
import numpy as np
import torch
from torch.utils.data import random_split
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from typing import Tuple, List, Optional
import energyflow as ef

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import global_mean_pool, global_max_pool


/home/jivnesh/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Preprocessing

In [7]:
def preprocess_jet(
    particles: np.ndarray,
    center_jet: bool = True,
    log_pt: bool = True,
    compute_pt_frac: bool = True
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Preprocess a single jet's particle features.

    Args:
        particles: (M, 4) array with columns [pT, rapidity, phi, pdgid]
        center_jet: Whether to center the jet at (eta=0, phi=0)
        log_pt: Whether to use log(pT) instead of pT
        compute_pt_frac: Whether to compute pT fraction

    Returns:
        coords: (N, 2) array of (delta_eta, delta_phi) for graph construction
        features: (N, D) array of node features
    """
    mask = particles[:, 0] > 0
    particles = particles[mask]

    if len(particles) == 0:
        return np.zeros((1, 2)), np.zeros((1, 5))

    pt = particles[:, 0]
    eta = particles[:, 1]
    phi = particles[:, 2]

    if center_jet:
        pt_sum = pt.sum()
        eta_center = (pt * eta).sum() / pt_sum
        phi_center = (pt * phi).sum() / pt_sum

        delta_eta = eta - eta_center
        delta_phi = phi - phi_center
        delta_phi = np.arctan2(np.sin(delta_phi), np.cos(delta_phi))
    else:
        delta_eta = eta
        delta_phi = phi

    coords = np.stack([delta_eta, delta_phi], axis=1)

    energy = pt * np.cosh(eta)

    feature_list = [delta_eta, delta_phi]

    if log_pt:
        feature_list.append(np.log(pt + 1e-8))
        feature_list.append(np.log(energy + 1e-8))
    else:
        feature_list.append(pt)
        feature_list.append(energy)

    if compute_pt_frac:
        pt_frac = pt / pt.sum()
        feature_list.append(pt_frac)

    features = np.stack(feature_list, axis=1)

    return coords.astype(np.float32), features.astype(np.float32)

In [8]:
def build_knn_graph(
    coords: torch.Tensor,
    k: int = 16,
    loop: bool = False
) -> torch.Tensor:
    """
    Build k-nearest neighbor graph from coordinates (no torch-cluster required).

    Args:
        coords: (N, 2) tensor of (eta, phi) coordinates
        k: Number of nearest neighbors
        loop: Whether to include self-loops

    Returns:
        edge_index: (2, E) tensor of edge indices
    """
    n = coords.size(0)

    if n <= 1:
        return torch.zeros((2, 0), dtype=torch.long)

    k_actual = min(k, n - 1) if not loop else min(k, n)

    if k_actual < 1:
        return torch.zeros((2, 0), dtype=torch.long)

    dist = torch.cdist(coords, coords)

    if not loop:
        dist.fill_diagonal_(float('inf'))

    _, knn_idx = dist.topk(k_actual, dim=1, largest=False)

    src = torch.arange(n).unsqueeze(1).expand(-1, k_actual).reshape(-1)
    dst = knn_idx.reshape(-1)

    edge_index = torch.stack([src, dst])

    return edge_index

In [9]:
def compute_edge_features(
    coords: torch.Tensor,
    features: torch.Tensor,
    edge_index: torch.Tensor
) -> torch.Tensor:
    """
    Compute edge features for GAT model.

    Args:
        coords: (N, 2) tensor of coordinates
        features: (N, D) tensor of node features
        edge_index: (2, E) tensor of edge indices

    Returns:
        edge_attr: (E, 3) tensor with (delta_R, delta_eta, delta_phi)
    """
    src, dst = edge_index[0], edge_index[1]

    delta_eta = coords[dst, 0] - coords[src, 0]
    delta_phi = coords[dst, 1] - coords[src, 1]

    delta_R = torch.sqrt(delta_eta**2 + delta_phi**2)

    edge_attr = torch.stack([delta_R, delta_eta, delta_phi], dim=1)

    return edge_attr

In [10]:
def normalize_features(
    features: np.ndarray,
    mean: Optional[np.ndarray] = None,
    std: Optional[np.ndarray] = None
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Normalize features to zero mean and unit variance.

    Args:
        features: (N, D) array of features
        mean: Pre-computed mean (for test set)
        std: Pre-computed std (for test set)

    Returns:
        normalized: Normalized features
        mean: Feature means
        std: Feature stds
    """
    if mean is None:
        mean = features.mean(axis=0)
    if std is None:
        std = features.std(axis=0)
        std[std < 1e-8] = 1.0

    normalized = (features - mean) / std

    return normalized, mean, std

## Dataset


In [11]:
class JetGraphDataset(Dataset):
    """
    Dataset that converts jets to graphs for GNN classification.
    """

    def __init__(
        self,
        X: np.ndarray,
        y: np.ndarray,
        k: int = 16,
        compute_edge_attr: bool = True,
        transform=None,
        pre_transform=None
    ):
        """
        Args:
            X: (N, M, 4) array of jet constituents
            y: (N,) array of labels (0=gluon, 1=quark)
            k: Number of neighbors for k-NN graph
            compute_edge_attr: Whether to compute edge features
        """
        super().__init__(None, transform, pre_transform)
        self.X = X
        self.y = y
        self.k = k
        self.compute_edge_attr = compute_edge_attr

    def len(self) -> int:
        return len(self.y)

    def get(self, idx: int) -> Data:
        """Get a single graph from the dataset."""
        particles = self.X[idx]
        label = self.y[idx]

        coords, features = preprocess_jet(particles)

        coords_tensor = torch.tensor(coords, dtype=torch.float32)
        features_tensor = torch.tensor(features, dtype=torch.float32)

        edge_index = build_knn_graph(coords_tensor, k=self.k)

        edge_attr = None
        if self.compute_edge_attr and edge_index.size(1) > 0:
            edge_attr = compute_edge_features(coords_tensor, features_tensor, edge_index)

        data = Data(
            x=features_tensor,
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=torch.tensor([label], dtype=torch.long),
            coords=coords_tensor
        )

        return data


In [12]:
def load_quark_gluon_data(
    num_data: int = 100000,
    cache_dir: str = '~/.energyflow',
    with_bc: bool = False
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load quark/gluon jet data using EnergyFlow package.

    Args:
        num_data: Number of jets to load
        cache_dir: Directory to cache downloaded data
        with_bc: Whether to include b/c jets

    Returns:
        X: (N, M, 4) array of jet constituents
        y: (N,) array of labels
    """
    print(f"Loading {num_data} jets from EnergyFlow...")

    X, y = ef.qg_jets.load(num_data=num_data, cache_dir=cache_dir, with_bc=with_bc)

    print(f"Loaded {len(y)} jets")
    print(f"  - Quarks: {(y == 1).sum()}")
    print(f"  - Gluons: {(y == 0).sum()}")
    print(f"  - Max multiplicity: {X.shape[1]}")

    return X, y


In [13]:

def create_data_loaders(
    X: np.ndarray,
    y: np.ndarray,
    k: int = 16,
    batch_size: int = 256,  # Increased for A100 GPU
    train_ratio: float = 0.7,
    val_ratio: float = 0.15,
    compute_edge_attr: bool = True,
    num_workers: int = 8,  # Optimized for multi-core CPU
    seed: int = 42
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """
    Create train/val/test data loaders.

    Args:
        X: Jet constituents array
        y: Labels array
        k: Number of neighbors for k-NN
        batch_size: Batch size for data loaders
        train_ratio: Fraction of data for training
        val_ratio: Fraction of data for validation
        compute_edge_attr: Whether to compute edge features
        num_workers: Number of workers for data loading
        seed: Random seed for splitting

    Returns:
        train_loader, val_loader, test_loader
    """
    dataset = JetGraphDataset(X, y, k=k, compute_edge_attr=compute_edge_attr)

    n_total = len(dataset)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)
    n_test = n_total - n_train - n_val

    generator = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset, test_dataset = random_split(
        dataset, [n_train, n_val, n_test], generator=generator
    )

    print(f"Dataset splits: train={n_train}, val={n_val}, test={n_test}")

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,  # GPU optimization
        persistent_workers=True if num_workers > 0 else False  # Keep workers alive
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True if num_workers > 0 else False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True if num_workers > 0 else False
    )

    return train_loader, val_loader, test_loader

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import global_mean_pool, global_max_pool

### Particle Net

In [14]:
def knn_graph_batch(x: torch.Tensor, k: int, batch: torch.Tensor) -> torch.Tensor:
    """
    Compute k-NN graph for batched point clouds without torch-cluster.

    Args:
        x: (N, D) node features
        k: number of neighbors
        batch: (N,) batch assignment

    Returns:
        edge_index: (2, N*k) edge indices
    """
    device = x.device
    batch_size = batch.max().item() + 1

    edge_sources = []
    edge_targets = []

    for b in range(batch_size):
        mask = (batch == b)
        indices = torch.where(mask)[0]
        x_b = x[mask]
        n = x_b.size(0)

        if n <= 1:
            continue

        k_actual = min(k, n - 1)

        dist = torch.cdist(x_b, x_b)
        dist.fill_diagonal_(float('inf'))

        _, knn_idx = dist.topk(k_actual, dim=1, largest=False)

        src = indices.unsqueeze(1).expand(-1, k_actual).reshape(-1)
        dst = indices[knn_idx.reshape(-1)]

        edge_sources.append(src)
        edge_targets.append(dst)

    if len(edge_sources) == 0:
        return torch.zeros((2, 0), dtype=torch.long, device=device)

    edge_index = torch.stack([
        torch.cat(edge_sources),
        torch.cat(edge_targets)
    ])

    return edge_index

In [15]:
class EdgeConvBlock(nn.Module):
    """
    EdgeConv block with MLP and batch normalization.
    Uses custom k-NN implementation (no torch-cluster dependency).
    """

    def __init__(self, in_channels: int, out_channels: int, k: int = 16):
        super().__init__()
        self.k = k
        self.in_channels = in_channels

        self.mlp = nn.Sequential(
            nn.Linear(2 * in_channels, out_channels),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Linear(out_channels, out_channels),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Linear(out_channels, out_channels),
            nn.BatchNorm1d(out_channels),
            nn.ReLU()
        )

        self.shortcut = nn.Linear(in_channels, out_channels) if in_channels != out_channels else nn.Identity()

    def forward(self, x: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through EdgeConv block.

        Args:
            x: (N, C) node features
            batch: (N,) batch indices

        Returns:
            x: (N, C') updated node features
        """
        edge_index = knn_graph_batch(x, self.k, batch)

        if edge_index.size(1) == 0:
            return self.shortcut(x)

        src, dst = edge_index[0], edge_index[1]

        x_src = x[src]
        x_dst = x[dst]
        edge_features = torch.cat([x_src, x_dst - x_src], dim=1)

        edge_out = self.mlp(edge_features)

        # Fix: Create tensor with same dtype as edge_out
        x_out = torch.zeros(x.size(0), edge_out.size(1), dtype=edge_out.dtype, device=x.device)

        x_out = x_out.scatter_reduce(
            0,
            src.unsqueeze(1).expand(-1, edge_out.size(1)),
            edge_out,
            reduce='amax',
            include_self=False
        )

        x_shortcut = self.shortcut(x)

        return x_out + x_shortcut

In [16]:
class ParticleNet(nn.Module):
    """
    ParticleNet architecture for jet classification.

    Uses dynamic k-NN graph construction in learned feature space
    with EdgeConv message passing.
    """

    def __init__(
        self,
        input_dim: int = 5,
        hidden_dims: Tuple[int, ...] = (64, 128, 256),
        k: int = 16,
        num_classes: int = 2,
        dropout: float = 0.3
    ):
        """
        Args:
            input_dim: Number of input node features
            hidden_dims: Hidden dimensions for each EdgeConv block
            k: Number of neighbors for k-NN
            num_classes: Number of output classes
            dropout: Dropout probability
        """
        super().__init__()

        self.input_dim = input_dim
        self.k = k

        self.input_bn = nn.BatchNorm1d(input_dim)

        self.edge_convs = nn.ModuleList()
        in_channels = input_dim

        for out_channels in hidden_dims:
            self.edge_convs.append(
                EdgeConvBlock(in_channels, out_channels, k=k)
            )
            in_channels = out_channels

        pool_dim = hidden_dims[-1] * 2

        self.classifier = nn.Sequential(
            nn.Linear(pool_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, data) -> torch.Tensor:
        """
        Forward pass through ParticleNet.

        Args:
            data: PyG Data object with x, batch attributes

        Returns:
            logits: (B, num_classes) classification logits
        """
        x, batch = data.x, data.batch

        x = self.input_bn(x)

        for edge_conv in self.edge_convs:
            x = edge_conv(x, batch)

        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=1)

        logits = self.classifier(x)

        return logits

    def get_embeddings(self, data) -> torch.Tensor:
        """Get node embeddings before global pooling."""
        x, batch = data.x, data.batch

        x = self.input_bn(x)

        for edge_conv in self.edge_convs:
            x = edge_conv(x, batch)

        return x


### GAT Classifier


In [17]:
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool

In [18]:
class GATBlock(nn.Module):
    """
    GAT block with multi-head attention and residual connection.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        heads: int = 4,
        concat: bool = True,
        edge_dim: Optional[int] = None,
        dropout: float = 0.1
    ):
        super().__init__()

        self.concat = concat
        actual_out = out_channels * heads if concat else out_channels

        self.gat = GATv2Conv(
            in_channels,
            out_channels,
            heads=heads,
            concat=concat,
            edge_dim=edge_dim,
            dropout=dropout,
            add_self_loops=True
        )

        self.bn = nn.BatchNorm1d(actual_out)

        if in_channels != actual_out:
            self.shortcut = nn.Linear(in_channels, actual_out)
        else:
            self.shortcut = nn.Identity()

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: Optional[torch.Tensor] = None,
        return_attention: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Forward pass through GAT block.

        Args:
            x: (N, C) node features
            edge_index: (2, E) edge indices
            edge_attr: (E, D) edge features (optional)
            return_attention: Whether to return attention weights

        Returns:
            x: (N, C') updated node features
            attention: (E, H) attention weights if return_attention=True
        """
        if return_attention:
            x_out, attention = self.gat(
                x, edge_index, edge_attr=edge_attr, return_attention_weights=True
            )
        else:
            x_out = self.gat(x, edge_index, edge_attr=edge_attr)
            attention = None

        x_out = self.bn(x_out)
        x_out = F.relu(x_out)

        x_shortcut = self.shortcut(x)
        x_out = x_out + x_shortcut

        return x_out, attention


In [19]:
class GATClassifier(nn.Module):
    """
    GAT-based classifier for quark/gluon jet classification.

    Uses static k-NN graph with attention mechanism for
    adaptive neighbor aggregation.
    """

    def __init__(
        self,
        input_dim: int = 5,
        hidden_dim: int = 64,
        num_layers: int = 3,
        heads: int = 4,
        edge_dim: int = 3,
        num_classes: int = 2,
        dropout: float = 0.3
    ):
        """
        Args:
            input_dim: Number of input node features
            hidden_dim: Hidden dimension per attention head
            num_layers: Number of GAT layers
            heads: Number of attention heads
            edge_dim: Number of edge features (set to None to disable)
            num_classes: Number of output classes
            dropout: Dropout probability
        """
        super().__init__()

        self.input_dim = input_dim
        self.edge_dim = edge_dim

        self.input_bn = nn.BatchNorm1d(input_dim)

        self.gat_blocks = nn.ModuleList()

        in_channels = input_dim
        for i in range(num_layers):
            is_last = (i == num_layers - 1)
            concat = not is_last

            self.gat_blocks.append(
                GATBlock(
                    in_channels,
                    hidden_dim,
                    heads=heads,
                    concat=concat,
                    edge_dim=edge_dim,
                    dropout=dropout if not is_last else 0.0
                )
            )

            in_channels = hidden_dim * heads if concat else hidden_dim

        pool_dim = hidden_dim * 2

        self.classifier = nn.Sequential(
            nn.Linear(pool_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(
        self,
        data,
        return_attention: bool = False
    ) -> Tuple[torch.Tensor, Optional[list]]:
        """
        Forward pass through GAT classifier.

        Args:
            data: PyG Data object with x, edge_index, edge_attr, batch
            return_attention: Whether to return attention weights

        Returns:
            logits: (B, num_classes) classification logits
            attentions: List of attention weights per layer (if requested)
        """
        x = data.x
        edge_index = data.edge_index
        edge_attr = data.edge_attr if hasattr(data, 'edge_attr') else None
        batch = data.batch

        x = self.input_bn(x)

        attentions = []
        for gat_block in self.gat_blocks:
            x, attn = gat_block(x, edge_index, edge_attr, return_attention)
            if return_attention and attn is not None:
                attentions.append(attn)

        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=1)

        logits = self.classifier(x)

        if return_attention:
            return logits, attentions
        return logits

    def get_attention_weights(self, data) -> list:
        """Get attention weights for visualization."""
        _, attentions = self.forward(data, return_attention=True)
        return attentions


## Training Utils


In [20]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score,
    confusion_matrix, classification_report
)
from typing import Dict, Tuple, List, Optional

In [21]:
def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: np.ndarray
) -> Dict[str, float]:
    """
    Compute classification metrics.

    Args:
        y_true: Ground truth labels
        y_pred: Predicted labels
        y_prob: Prediction probabilities for positive class

    Returns:
        Dictionary of metrics
    """
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_prob)
    }

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    metrics['quark_precision'] = tp / (tp + fp) if (tp + fp) > 0 else 0
    metrics['quark_recall'] = tp / (tp + fn) if (tp + fn) > 0 else 0
    metrics['gluon_precision'] = tn / (tn + fn) if (tn + fn) > 0 else 0
    metrics['gluon_recall'] = tn / (tn + fp) if (tn + fp) > 0 else 0

    return metrics


In [22]:
def plot_roc_curve(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    model_name: str = "Model",
    ax: Optional[plt.Axes] = None,
    save_path: Optional[str] = None
) -> Tuple[plt.Figure, plt.Axes]:
    """
    Plot ROC curve.

    Args:
        y_true: Ground truth labels
        y_prob: Prediction probabilities
        model_name: Name for legend
        ax: Matplotlib axes (creates new if None)
        save_path: Path to save figure

    Returns:
        Figure and axes
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    else:
        fig = ax.figure

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)

    ax.plot(fpr, tpr, label=f'{model_name} (AUC = {auc:.4f})', linewidth=2)
    ax.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)

    ax.set_xlabel('False Positive Rate (Gluon Misidentification)', fontsize=12)
    ax.set_ylabel('True Positive Rate (Quark Efficiency)', fontsize=12)
    ax.set_title('ROC Curve for Quark/Gluon Classification', fontsize=14)
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig, ax


In [23]:
def plot_roc_comparison(
    results: Dict[str, Tuple[np.ndarray, np.ndarray]],
    save_path: Optional[str] = None
) -> Tuple[plt.Figure, plt.Axes]:
    """
    Plot ROC curves for multiple models.

    Args:
        results: Dict mapping model names to (y_true, y_prob) tuples
        save_path: Path to save figure

    Returns:
        Figure and axes
    """
    fig, ax = plt.subplots(figsize=(8, 6))

    colors = plt.cm.tab10.colors

    for i, (name, (y_true, y_prob)) in enumerate(results.items()):
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc = roc_auc_score(y_true, y_prob)
        ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})',
                linewidth=2, color=colors[i % len(colors)])

    ax.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)

    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Comparison: Quark/Gluon Classification', fontsize=14)
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig, ax

In [24]:
def plot_training_history(
    history: Dict[str, List[float]],
    model_name: str = "Model",
    save_path: Optional[str] = None
) -> Tuple[plt.Figure, plt.Axes]:
    """
    Plot training history (loss and metrics over epochs).

    Args:
        history: Dictionary with 'train_loss', 'val_loss', 'train_auc', 'val_auc'
        model_name: Model name for title
        save_path: Path to save figure

    Returns:
        Figure and axes
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    epochs = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title(f'{model_name}: Training Loss', fontsize=14)
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)

    if 'train_auc' in history:
        axes[1].plot(epochs, history['train_auc'], 'b-', label='Train AUC', linewidth=2)
        axes[1].plot(epochs, history['val_auc'], 'r-', label='Val AUC', linewidth=2)
        axes[1].set_xlabel('Epoch', fontsize=12)
        axes[1].set_ylabel('AUC', fontsize=12)
        axes[1].set_title(f'{model_name}: AUC Score', fontsize=14)
        axes[1].legend(fontsize=10)
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig, axes

## Trainer Class

In [25]:
import torch.optim as optim
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import os

In [26]:
class Trainer:
    """
    Trainer class for GNN jet classifiers with GPU optimizations.
    """

    def __init__(
        self,
        model: nn.Module,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu',
        learning_rate: float = 1e-3,
        weight_decay: float = 1e-4,
        use_amp: bool = True  # Automatic Mixed Precision for A100
    ):
        """
        Args:
            model: PyTorch model to train
            device: Device to train on
            learning_rate: Initial learning rate
            weight_decay: L2 regularization weight
            use_amp: Use automatic mixed precision (FP16) for faster training
        """
        self.model = model.to(device)
        self.device = device
        self.use_amp = use_amp and device == 'cuda'

        # Enable TF32 for A100 GPUs (faster matmul)
        if torch.cuda.is_available():
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
            torch.backends.cudnn.benchmark = True  # Auto-tune kernels

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.AdamW(  # AdamW is better than Adam
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode='max',
            factor=0.5,
            patience=5
        )

        # Gradient scaler for mixed precision
        self.scaler = torch.cuda.amp.GradScaler() if self.use_amp else None

        self.history = {
            'train_loss': [],
            'val_loss': [],
            'train_auc': [],
            'val_auc': []
        }

    def train_epoch(self, train_loader: DataLoader, epoch: int, total_epochs: int) -> Tuple[float, float]:
        """
        Train for one epoch with detailed progress and mixed precision.

        Returns:
            Average loss and AUC for the epoch
        """
        self.model.train()
        total_loss = 0
        all_labels = []
        all_probs = []
        n_batches = len(train_loader)

        pbar = tqdm(train_loader,
                    desc=f"Epoch {epoch+1}/{total_epochs} [Train]",
                    leave=False,
                    ncols=100)

        for batch_idx, data in enumerate(pbar):
            data = data.to(self.device, non_blocking=True)  # Async transfer

            self.optimizer.zero_grad(set_to_none=True)  # Faster than zero_grad()

            # Mixed precision forward pass
            if self.use_amp:
                with torch.cuda.amp.autocast():
                    out = self.model(data)
                    if isinstance(out, tuple):
                        out = out[0]
                    loss = self.criterion(out, data.y.view(-1))
                
                # Scaled backward pass
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                out = self.model(data)
                if isinstance(out, tuple):
                    out = out[0]
                loss = self.criterion(out, data.y.view(-1))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()

            total_loss += loss.item() * data.num_graphs

            probs = torch.softmax(out, dim=1)[:, 1].detach().cpu().numpy()
            labels = data.y.view(-1).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels)

            running_loss = total_loss / ((batch_idx + 1) * train_loader.batch_size)
            pbar.set_postfix({'loss': f'{running_loss:.4f}'})

        avg_loss = total_loss / len(train_loader.dataset)
        auc = roc_auc_score(all_labels, all_probs)

        return avg_loss, auc

    @torch.no_grad()
    def evaluate(self, loader: DataLoader, desc: str = "Eval") -> Tuple[float, float, np.ndarray, np.ndarray]:
        """
        Evaluate the model with progress bar and mixed precision.

        Returns:
            Average loss, AUC, true labels, and prediction probabilities
        """
        self.model.eval()
        total_loss = 0
        all_labels = []
        all_probs = []

        pbar = tqdm(loader, desc=f"[{desc}]", leave=False, ncols=100)

        for data in pbar:
            data = data.to(self.device, non_blocking=True)

            if self.use_amp:
                with torch.cuda.amp.autocast():
                    out = self.model(data)
                    if isinstance(out, tuple):
                        out = out[0]
                    loss = self.criterion(out, data.y.view(-1))
            else:
                out = self.model(data)
                if isinstance(out, tuple):
                    out = out[0]
                loss = self.criterion(out, data.y.view(-1))

            total_loss += loss.item() * data.num_graphs

            probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            labels = data.y.view(-1).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels)

        avg_loss = total_loss / len(loader.dataset)
        all_labels = np.array(all_labels)
        all_probs = np.array(all_probs)
        auc = roc_auc_score(all_labels, all_probs)

        return avg_loss, auc, all_labels, all_probs

    def train(
        self,
        train_loader: DataLoader,
        val_loader: DataLoader,
        epochs: int = 50,
        early_stopping_patience: int = 10,
        save_path: Optional[str] = None,
        model_name: str = "model"
    ) -> Dict[str, List[float]]:
        """
        Full training loop with validation.

        Args:
            train_loader: Training data loader
            val_loader: Validation data loader
            epochs: Maximum number of epochs
            early_stopping_patience: Patience for early stopping
            save_path: Directory to save best model
            model_name: Name for saved model file

        Returns:
            Training history dictionary
        """
        best_val_auc = 0
        patience_counter = 0

        print(f"\nStarting training for {epochs} epochs...")
        print(f"Device: {self.device}, Mixed Precision: {self.use_amp}")
        print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
        print("-" * 80)

        for epoch in range(epochs):
            train_loss, train_auc = self.train_epoch(train_loader, epoch, epochs)
            val_loss, val_auc, _, _ = self.evaluate(val_loader, desc="Val")

            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['train_auc'].append(train_auc)
            self.history['val_auc'].append(val_auc)

            self.scheduler.step(val_auc)

            lr = self.optimizer.param_groups[0]['lr']
            improved = "*" if val_auc > best_val_auc else ""
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Train Loss: {train_loss:.4f}, AUC: {train_auc:.4f} | "
                  f"Val Loss: {val_loss:.4f}, AUC: {val_auc:.4f} {improved} | "
                  f"LR: {lr:.2e}")

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                patience_counter = 0

                if save_path:
                    os.makedirs(save_path, exist_ok=True)
                    torch.save(
                        self.model.state_dict(),
                        os.path.join(save_path, f'{model_name}_best.pt')
                    )
            else:
                patience_counter += 1

                if patience_counter >= early_stopping_patience:
                    print(f"\nEarly stopping at epoch {epoch + 1}")
                    break

        print(f"\nBest validation AUC: {best_val_auc:.4f}")

        return self.history

    def load_best_model(self, save_path: str, model_name: str = "model"):
        """Load the best saved model."""
        path = os.path.join(save_path, f'{model_name}_best.pt')
        self.model.load_state_dict(torch.load(path, map_location=self.device))
        print(f"Loaded best model from {path}")

## Visualization Utils


In [27]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

In [28]:
def plot_jet(
    coords: np.ndarray,
    pt: np.ndarray,
    label: int,
    ax: Optional[plt.Axes] = None,
    title: Optional[str] = None,
    save_path: Optional[str] = None
) -> plt.Figure:
    """
    Plot a jet in (eta, phi) space.

    Args:
        coords: (N, 2) array of (delta_eta, delta_phi)
        pt: (N,) array of pT values
        label: Jet label (0=gluon, 1=quark)
        ax: Matplotlib axes
        title: Plot title
        save_path: Path to save figure

    Returns:
        Figure
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
    else:
        fig = ax.figure

    sizes = 50 * (pt / pt.max()) ** 0.5
    sizes = np.clip(sizes, 5, 200)

    scatter = ax.scatter(
        coords[:, 0], coords[:, 1],
        s=sizes, c=np.log(pt + 1e-8),
        cmap='viridis', alpha=0.7, edgecolors='black', linewidth=0.5
    )

    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('log(pT)', fontsize=10)

    label_str = "Quark" if label == 1 else "Gluon"
    if title is None:
        title = f"{label_str} Jet (N={len(pt)} particles)"

    ax.set_xlabel('Δη', fontsize=12)
    ax.set_ylabel('Δφ', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig


In [29]:
def plot_jet_with_graph(
    coords: np.ndarray,
    pt: np.ndarray,
    edge_index: np.ndarray,
    label: int,
    ax: Optional[plt.Axes] = None,
    title: Optional[str] = None,
    save_path: Optional[str] = None
) -> plt.Figure:
    """
    Plot a jet with k-NN graph edges.

    Args:
        coords: (N, 2) array of coordinates
        pt: (N,) array of pT values
        edge_index: (2, E) array of edge indices
        label: Jet label
        ax: Matplotlib axes
        title: Plot title
        save_path: Path to save figure
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 10))
    else:
        fig = ax.figure

    for i in range(edge_index.shape[1]):
        src, dst = edge_index[0, i], edge_index[1, i]
        ax.plot(
            [coords[src, 0], coords[dst, 0]],
            [coords[src, 1], coords[dst, 1]],
            'gray', alpha=0.2, linewidth=0.5
        )

    sizes = 100 * (pt / pt.max()) ** 0.5
    sizes = np.clip(sizes, 10, 300)

    scatter = ax.scatter(
        coords[:, 0], coords[:, 1],
        s=sizes, c=np.log(pt + 1e-8),
        cmap='viridis', alpha=0.8, edgecolors='black', linewidth=0.5, zorder=10
    )

    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('log(pT)', fontsize=10)

    label_str = "Quark" if label == 1 else "Gluon"
    if title is None:
        title = f"{label_str} Jet Graph (N={len(pt)}, E={edge_index.shape[1]})"

    ax.set_xlabel('Δη', fontsize=12)
    ax.set_ylabel('Δφ', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig

In [30]:
def plot_attention_weights(
    coords: np.ndarray,
    pt: np.ndarray,
    edge_index: np.ndarray,
    attention_weights: np.ndarray,
    label: int,
    ax: Optional[plt.Axes] = None,
    title: Optional[str] = None,
    save_path: Optional[str] = None
) -> plt.Figure:
    """
    Plot jet with attention-weighted edges.

    Args:
        coords: (N, 2) array of coordinates
        pt: (N,) array of pT
        edge_index: (2, E) edge indices
        attention_weights: (E,) attention weights
        label: Jet label
        ax: Matplotlib axes
        title: Plot title
        save_path: Path to save figure
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 10))
    else:
        fig = ax.figure

    attn_norm = (attention_weights - attention_weights.min()) / (attention_weights.max() - attention_weights.min() + 1e-8)

    for i in range(edge_index.shape[1]):
        src, dst = edge_index[0, i], edge_index[1, i]
        alpha = 0.1 + 0.8 * attn_norm[i]
        width = 0.5 + 3 * attn_norm[i]
        ax.plot(
            [coords[src, 0], coords[dst, 0]],
            [coords[src, 1], coords[dst, 1]],
            color=plt.cm.Reds(attn_norm[i]),
            alpha=alpha, linewidth=width
        )

    sizes = 100 * (pt / pt.max()) ** 0.5
    sizes = np.clip(sizes, 10, 300)

    ax.scatter(
        coords[:, 0], coords[:, 1],
        s=sizes, c='steelblue',
        alpha=0.8, edgecolors='black', linewidth=0.5, zorder=10
    )

    label_str = "Quark" if label == 1 else "Gluon"
    if title is None:
        title = f"{label_str} Jet - Attention Weights"

    ax.set_xlabel('Δη', fontsize=12)
    ax.set_ylabel('Δφ', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig

In [31]:
def plot_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    model_name: str = "Model",
    save_path: Optional[str] = None
) -> plt.Figure:
    """
    Plot confusion matrix.

    Args:
        y_true: True labels
        y_pred: Predicted labels
        model_name: Model name for title
        save_path: Path to save figure
    """
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(8, 6))

    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Gluon', 'Quark'],
        yticklabels=['Gluon', 'Quark'],
        ax=ax
    )

    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(f'{model_name}: Confusion Matrix', fontsize=14)

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig

In [32]:
def plot_multiplicity_distribution(
    X: np.ndarray,
    y: np.ndarray,
    save_path: Optional[str] = None
) -> plt.Figure:
    """
    Plot particle multiplicity distribution for quarks vs gluons.

    Args:
        X: (N, M, 4) jet data
        y: (N,) labels
        save_path: Path to save figure
    """
    multiplicities = (X[:, :, 0] > 0).sum(axis=1)

    fig, ax = plt.subplots(figsize=(10, 6))

    quark_mult = multiplicities[y == 1]
    gluon_mult = multiplicities[y == 0]

    bins = np.arange(0, multiplicities.max() + 5, 5)

    ax.hist(quark_mult, bins=bins, alpha=0.6, label=f'Quark (mean={quark_mult.mean():.1f})',
            density=True, color='blue')
    ax.hist(gluon_mult, bins=bins, alpha=0.6, label=f'Gluon (mean={gluon_mult.mean():.1f})',
            density=True, color='red')

    ax.set_xlabel('Particle Multiplicity', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title('Particle Multiplicity: Quark vs Gluon Jets', fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig


## Training Loop

In [33]:
import argparse

In [34]:
def set_seed(seed: int):
    """Set random seeds for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [35]:
def train_particlenet(train_loader, val_loader, test_loader, args, device):
    """Train ParticleNet model."""
    print("\n" + "="*60)
    print("Training ParticleNet (Dynamic EdgeConv)")
    print("="*60)

    model = ParticleNet(
        input_dim=5,
        hidden_dims=(64, 128, 256),
        k=args.k,
        num_classes=2,
        dropout=args.dropout
    )

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    trainer = Trainer(model, device=device, learning_rate=args.lr)

    history = trainer.train(
        train_loader, val_loader,
        epochs=args.epochs,
        early_stopping_patience=args.patience,
        save_path=args.save_dir,
        model_name='particlenet'
    )

    trainer.load_best_model(args.save_dir, 'particlenet')
    test_loss, test_auc, y_true, y_prob = trainer.evaluate(test_loader)
    y_pred = (y_prob > 0.5).astype(int)

    print(f"\nParticleNet Test Results:")
    print(f"  Loss: {test_loss:.4f}")
    print(f"  AUC: {test_auc:.4f}")

    metrics = compute_metrics(y_true, y_pred, y_prob)
    print(f"  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Quark Precision: {metrics['quark_precision']:.4f}")
    print(f"  Quark Recall: {metrics['quark_recall']:.4f}")

    return model, history, (y_true, y_prob), metrics

In [36]:
def train_gat(train_loader, val_loader, test_loader, args, device):
    """Train GAT model."""
    print("\n" + "="*60)
    print("Training GAT Classifier (GATv2Conv)")
    print("="*60)

    model = GATClassifier(
        input_dim=5,
        hidden_dim=64,
        num_layers=3,
        heads=4,
        edge_dim=3,
        num_classes=2,
        dropout=args.dropout
    )

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    trainer = Trainer(model, device=device, learning_rate=args.lr)

    history = trainer.train(
        train_loader, val_loader,
        epochs=args.epochs,
        early_stopping_patience=args.patience,
        save_path=args.save_dir,
        model_name='gat'
    )

    trainer.load_best_model(args.save_dir, 'gat')
    test_loss, test_auc, y_true, y_prob = trainer.evaluate(test_loader)
    y_pred = (y_prob > 0.5).astype(int)

    print(f"\nGAT Test Results:")
    print(f"  Loss: {test_loss:.4f}")
    print(f"  AUC: {test_auc:.4f}")

    metrics = compute_metrics(y_true, y_pred, y_prob)
    print(f"  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Quark Precision: {metrics['quark_precision']:.4f}")
    print(f"  Quark Recall: {metrics['quark_recall']:.4f}")

    return model, history, (y_true, y_prob), metrics

In [38]:
def train(model = "both", num_data = 100000, batch_size = 128, epochs = 50, lr = 1e-3, k = 16, dropout = 0.3, patience = 10, save_dir = 'results', seed = 42):
    from types import SimpleNamespace
    args = {'model': model, 'num_data': num_data, 'batch_size' : batch_size, 'epochs' : epochs, 'lr' : lr, 'k' : k, 'dropout': dropout, 'patience': patience, 'save_dir': save_dir, 'seed': seed}
    args = SimpleNamespace(**args)
    os.makedirs(args.save_dir, exist_ok=True)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    print("\n" + "="*60)
    print("Loading Data")
    print("="*60)

    X, y = load_quark_gluon_data(num_data=args.num_data)

    fig = plot_multiplicity_distribution(X, y)
    fig.savefig(os.path.join(args.save_dir, 'multiplicity_distribution.png'),
                dpi=150, bbox_inches='tight')
    plt.close(fig)

    compute_edge = (args.model in ['gat', 'both'])

    train_loader, val_loader, test_loader = create_data_loaders(
        X, y,
        k=args.k,
        batch_size=args.batch_size,
        compute_edge_attr=compute_edge,
        seed=args.seed
    )

    results = {}

    if args.model in ['particlenet', 'both']:
        if not compute_edge:
            train_loader_pn, val_loader_pn, test_loader_pn = create_data_loaders(
                X, y, k=args.k, batch_size=args.batch_size,
                compute_edge_attr=False, seed=args.seed
            )
        else:
            train_loader_pn, val_loader_pn, test_loader_pn = train_loader, val_loader, test_loader

        pn_model, pn_history, pn_results, pn_metrics = train_particlenet(
            train_loader_pn, val_loader_pn, test_loader_pn, args, device
        )
        results['ParticleNet'] = pn_results

        plot_training_history(pn_history, 'ParticleNet',
                              os.path.join(args.save_dir, 'particlenet_history.png'))
        plt.close()

        plot_confusion_matrix(pn_results[0], (pn_results[1] > 0.5).astype(int),
                              'ParticleNet',
                              os.path.join(args.save_dir, 'particlenet_confusion.png'))
        plt.close()

    if args.model in ['gat', 'both']:
        gat_model, gat_history, gat_results, gat_metrics = train_gat(
            train_loader, val_loader, test_loader, args, device
        )
        results['GAT'] = gat_results

        plot_training_history(gat_history, 'GAT',
                              os.path.join(args.save_dir, 'gat_history.png'))
        plt.close()

        plot_confusion_matrix(gat_results[0], (gat_results[1] > 0.5).astype(int),
                              'GAT',
                              os.path.join(args.save_dir, 'gat_confusion.png'))
        plt.close()

    if len(results) > 0:
        fig, ax = plot_roc_comparison(results)
        fig.savefig(os.path.join(args.save_dir, 'roc_comparison.png'),
                    dpi=150, bbox_inches='tight')
        plt.close()

    print("\n" + "="*60)
    print("Training Complete")
    print("="*60)
    print(f"Results saved to: {args.save_dir}/")

    if len(results) == 2:
        print("\n--- Final Comparison ---")
        for name, (y_true, y_prob) in results.items():
            from sklearn.metrics import roc_auc_score
            auc = roc_auc_score(y_true, y_prob)
            acc = ((y_prob > 0.5).astype(int) == y_true).mean()
            print(f"{name}: AUC = {auc:.4f}, Accuracy = {acc:.4f}")

## Model Evaluation


In [39]:
def load_model(model_type: str, checkpoint_path: str, device: str):
    """Load a trained model."""
    if model_type == 'particlenet':
        model = ParticleNet(
            input_dim=5,
            hidden_dims=(64, 128, 256),
            k=16,
            num_classes=2,
            dropout=0.3
        )
    else:
        model = GATClassifier(
            input_dim=5,
            hidden_dim=64,
            num_layers=3,
            heads=4,
            edge_dim=3,
            num_classes=2,
            dropout=0.3
        )

    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model = model.to(device)
    model.eval()

    print(f"Loaded {model_type} from {checkpoint_path}")
    return model


In [40]:
@torch.no_grad()
def evaluate_model(model, test_loader, device):
    """Evaluate model on test set."""
    all_labels = []
    all_probs = []

    for data in test_loader:
        data = data.to(device)
        out = model(data)
        if isinstance(out, tuple):
            out = out[0]

        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        labels = data.y.view(-1).cpu().numpy()

        all_probs.extend(probs)
        all_labels.extend(labels)

    y_true = np.array(all_labels)
    y_prob = np.array(all_probs)
    y_pred = (y_prob > 0.5).astype(int)

    return y_true, y_pred, y_prob


In [41]:
def visualize_jets(model, test_loader, model_type, save_dir, device, num_jets=4):
    """Visualize sample jets with predictions."""
    os.makedirs(save_dir, exist_ok=True)

    model.eval()

    data_iter = iter(test_loader)
    batch = next(data_iter).to(device)

    with torch.no_grad():
        if model_type == 'gat':
            logits, attentions = model(batch, return_attention=True)
        else:
            logits = model(batch)

    probs = torch.softmax(logits, dim=1)

    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    axes = axes.flatten()

    ptr = 0
    for i in range(min(num_jets, batch.num_graphs)):
        mask = batch.batch == i
        coords = batch.coords[mask].cpu().numpy()
        x = batch.x[mask].cpu().numpy()
        pt = np.exp(x[:, 2])

        edge_mask = (batch.edge_index[0] >= ptr) & (batch.edge_index[0] < ptr + mask.sum())
        local_edges = batch.edge_index[:, edge_mask].cpu().numpy()
        local_edges = local_edges - ptr

        true_label = batch.y[i].item()
        pred_prob = probs[i, 1].item()
        pred_label = 1 if pred_prob > 0.5 else 0

        true_str = "Quark" if true_label == 1 else "Gluon"
        pred_str = "Quark" if pred_label == 1 else "Gluon"

        ax = axes[i]

        for j in range(local_edges.shape[1]):
            src, dst = local_edges[0, j], local_edges[1, j]
            if src < len(coords) and dst < len(coords):
                ax.plot(
                    [coords[src, 0], coords[dst, 0]],
                    [coords[src, 1], coords[dst, 1]],
                    'gray', alpha=0.2, linewidth=0.5
                )

        sizes = 80 * (pt / pt.max()) ** 0.5
        sizes = np.clip(sizes, 10, 200)

        scatter = ax.scatter(
            coords[:, 0], coords[:, 1],
            s=sizes, c=np.log(pt + 1e-8),
            cmap='viridis', alpha=0.8, edgecolors='black', linewidth=0.5, zorder=10
        )

        correct = "✓" if true_label == pred_label else "✗"
        ax.set_title(f"True: {true_str} | Pred: {pred_str} ({pred_prob:.2f}) {correct}", fontsize=12)
        ax.set_xlabel('Δη')
        ax.set_ylabel('Δφ')
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)

        ptr += mask.sum().item()

    plt.tight_layout()
    fig.savefig(os.path.join(save_dir, f'{model_type}_sample_jets.png'),
                dpi=150, bbox_inches='tight')
    plt.close()

    print(f"Saved sample jets visualization to {save_dir}/{model_type}_sample_jets.png")

In [42]:
def evaluate(model, checkpoint, visualize, num_data = 100000, batch_size = 128, k = 16, save_dir = 'results', seed = 42):
    from types import SimpleNamespace
    args = {'model': model, 'checkpoint': checkpoint, 'visualize': visualize, 'num_data': num_data, 'batch_size': batch_size, 'k': k, 'save_dir': save_dir, 'seed': seed}
    args = SimpleNamespace(**args)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    model = load_model(args.model, args.checkpoint, device)

    print("Loading test data...")
    X, y = load_quark_gluon_data(num_data=args.num_data)

    compute_edge = (args.model == 'gat')
    _, _, test_loader = create_data_loaders(
        X, y,
        k=args.k,
        batch_size=args.batch_size,
        compute_edge_attr=compute_edge,
        seed=args.seed
    )

    print("Evaluating model...")
    y_true, y_pred, y_prob = evaluate_model(model, test_loader, device)

    metrics = compute_metrics(y_true, y_pred, y_prob)

    print("\n" + "="*50)
    print(f"Evaluation Results: {args.model.upper()}")
    print("="*50)
    print(f"AUC Score:        {metrics['auc']:.4f}")
    print(f"Accuracy:         {metrics['accuracy']:.4f}")
    print(f"Quark Precision:  {metrics['quark_precision']:.4f}")
    print(f"Quark Recall:     {metrics['quark_recall']:.4f}")
    print(f"Gluon Precision:  {metrics['gluon_precision']:.4f}")
    print(f"Gluon Recall:     {metrics['gluon_recall']:.4f}")

    os.makedirs(args.save_dir, exist_ok=True)

    plot_roc_curve(y_true, y_prob, args.model.upper(),
                   save_path=os.path.join(args.save_dir, f'{args.model}_roc.png'))
    plt.close()

    plot_confusion_matrix(y_true, y_pred, args.model.upper(),
                          save_path=os.path.join(args.save_dir, f'{args.model}_confusion.png'))
    plt.close()

    if args.visualize:
        print("\nGenerating visualizations...")
        visualize_jets(model, test_loader, args.model, args.save_dir, device)

    print(f"\nResults saved to {args.save_dir}/")

## Training and Evaluating

In [109]:
train(model = "both", num_data = 20000, batch_size = 128, epochs = 20, lr = 1e-3, k = 16, dropout = 0.3, patience = 10, save_dir = 'results', seed = 42)

Using device: cuda

Loading Data
Loading 20000 jets from EnergyFlow...
Loaded 20000 jets
  - Quarks: 10076
  - Gluons: 9924
  - Max multiplicity: 139


/tmp/ipykernel_1318162/1911452813.py:46: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler() if self.use_amp else None


Dataset splits: train=14000, val=3000, test=3000

Training ParticleNet (Dynamic EdgeConv)
Total parameters: 465,612
Trainable parameters: 465,612

Starting training for 20 epochs...
Device: cuda, Mixed Precision: True
Train batches: 110, Val batches: 24
--------------------------------------------------------------------------------


Epoch 1/20 [Train]:   0%|                                                   | 0/110 [00:00<?, ?it/s]/tmp/ipykernel_1318162/1911452813.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/jivnesh/.local/lib/python3.10/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
[Val]:   0%|                                                                 | 0/24 [00:00<?, ?it/s]/tmp/ipykernel_1318162/1911452813.py:135: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch   1/20 | Train Loss: 0.4900, AUC: 0.8453 | Val Loss: 0.4875, AUC: 0.8688 * | LR: 1.00e-03


...

Epoch  15/20 | Train Loss: 0.4186, AUC: 0.8898 | Val Loss: 0.4762, AUC: 0.8705  | LR: 5.00e-04


Epoch 16/20 [Train]:   0%|                                                  | 0/110 [00:00<?, ?it/s]/tmp/ipykernel_1318162/1911452813.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/jivnesh/.local/lib/python3.10/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
[Val]:   0%|                                                                 | 0/24 [00:00<?, ?it/s]/tmp/ipykernel_1318162/1911452813.py:135: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch  16/20 | Train Loss: 0.4181, AUC: 0.8908 | Val Loss: 0.4602, AUC: 0.8715  | LR: 5.00e-04

Early stopping at epoch 16

Best validation AUC: 0.8748
Loaded best model from results/gat_best.pt


[Eval]:   0%|                                                                | 0/24 [00:00<?, ?it/s]/tmp/ipykernel_1318162/1911452813.py:135: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



GAT Test Results:
  Loss: 0.4419
  AUC: 0.8786
  Accuracy: 0.7993
  Quark Precision: 0.8429
  Quark Recall: 0.7377

Training Complete
Results saved to: results/

--- Final Comparison ---
ParticleNet: AUC = 0.8729, Accuracy = 0.7983
GAT: AUC = 0.8786, Accuracy = 0.7993


In [43]:
evaluate(model = "particlenet", checkpoint = "results/particlenet_best.pt", visualize=True, num_data = 20000, batch_size = 128, k = 16, save_dir = 'results', seed = 42)

Using device: cuda


Loaded particlenet from results/particlenet_best.pt
Loading test data...
Loading 20000 jets from EnergyFlow...
Loaded 20000 jets
  - Quarks: 10076
  - Gluons: 9924
  - Max multiplicity: 139
Dataset splits: train=14000, val=3000, test=3000
Evaluating model...

Evaluation Results: PARTICLENET
AUC Score:        0.8728
Accuracy:         0.7977
Quark Precision:  0.8337
Quark Recall:     0.7457
Gluon Precision:  0.7683
Gluon Recall:     0.8501

Generating visualizations...
Saved sample jets visualization to results/particlenet_sample_jets.png

Results saved to results/


In [44]:
evaluate(model = "gat", checkpoint = "results/gat_best.pt", visualize=True, num_data = 20000, batch_size = 128, k = 16, save_dir = 'results', seed = 42)

Using device: cuda
Loaded gat from results/gat_best.pt
Loading test data...
Loading 20000 jets from EnergyFlow...
Loaded 20000 jets
  - Quarks: 10076
  - Gluons: 9924
  - Max multiplicity: 139
Dataset splits: train=14000, val=3000, test=3000
Evaluating model...

Evaluation Results: GAT
AUC Score:        0.8786
Accuracy:         0.7990
Quark Precision:  0.8423
Quark Recall:     0.7377
Gluon Precision:  0.7650
Gluon Recall:     0.8608

Generating visualizations...
Saved sample jets visualization to results/gat_sample_jets.png

Results saved to results/
